# SIPTA Notebook: Ingestión de datos

Este notebook sirve como guía inicial para la ingesta de datos crudos en el proyecto SIPTA.

## Objetivos

- Registrar y versionar las fuentes de datos originales.
- Cargar archivos desde `data/raw` con pandas.
- Guardar copias de seguridad para reproducibilidad.

In [ ]:
from pathlib import Path
import logging

import pandas as pd
import requests

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROCESSED_DIR

## Función de carga reutilizable

Esta función conserva la carga genérica definida para la fase ETL.

In [ ]:
def load_raw_file(filename: str, file_type: str = 'csv') -> pd.DataFrame:
    path = RAW_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'No existe el archivo: {path}')

    if file_type.lower() == 'csv':
        return pd.read_csv(path, low_memory=False)
    if file_type.lower() == 'json':
        return pd.read_json(path)
    raise ValueError(f'Tipo no soportado: {file_type}')

In [26]:
import requests
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def download_to_raw(url: str, filename: str, subcarpeta: str = "") -> Path:
    # 1. Definir y crear el directorio destino
    directorio_destino = RAW_DIR / subcarpeta
    directorio_destino.mkdir(parents=True, exist_ok=True)
    
    # 2. Definir la ruta final del archivo
    filepath = directorio_destino / filename
    
    if filepath.exists():
        logging.info(f"El archivo {filename} ya existe en {directorio_destino}. Omitiendo.")
        return filepath
        
    try:
        logging.info(f"Descargando {filename} en la subcarpeta '{subcarpeta}'...")
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        filepath.write_bytes(response.content)
        logging.info(f"✅ Archivo guardado correctamente en: {filepath}")
        return filepath
        
    except Exception as e:
        logging.error(f"❌ Error al descargar {filename}: {e}")
        return None

# --- Ejecución estructurada ---
url_poblacion = "https://datosabiertos.bogota.gov.co/dataset/85bf790d-84d1-4eda-bd6f-40af62e71d95/resource/37e58cb3-c870-4608-8c37-ce45db0eb7c1/download/osb_demografia-poblacion-localidad.csv"
archivo_poblacion = "osb_demografia-poblacion-localidad.csv"
carpeta_tematica = "DEMOGRAFIA"

filepath_poblacion = download_to_raw(url_poblacion, archivo_poblacion, subcarpeta=carpeta_tematica)


INFO: El archivo osb_demografia-poblacion-localidad.csv ya existe en C:\DataJam_DataOlinguitos_Gen\data\raw\DEMOGRAFIA. Omitiendo.


## Cargar un dataset crudo

In [27]:
def load_raw_csv(filename: str, subcarpeta: str = "", separador: str = ';') -> pd.DataFrame:
    path = RAW_DIR / subcarpeta / filename
    assert path.exists(), f'No existe el archivo en la ruta: {path}'
    
    return pd.read_csv(path, sep=separador, low_memory=False, encoding='utf-8-sig')

# Prueba de carga desde la nueva ruta:
df_poblacion = load_raw_csv('osb_demografia-poblacion-localidad.csv', subcarpeta='DEMOGRAFIA')
df_poblacion.head()

,ANO,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD,SEXO,EDAD,CURSODEVIDA,GRUPOEDAD,POBLACION
0,2005,0,Bogotá,Hombres,6,Infancia,00 a 11,67184
1,2005,0,Bogotá,Hombres,7,Infancia,00 a 11,68940
2,2005,0,Bogotá,Hombres,8,Infancia,00 a 11,70568
3,2005,0,Bogotá,Hombres,9,Infancia,00 a 11,71189
4,2005,0,Bogotá,Hombres,10,Infancia,00 a 11,70398


In [28]:
print("=== DIMENSIONES DEL DATASET ===")
print(f"Filas: {df_poblacion.shape[0]}")
print(f"Columnas: {df_poblacion.shape[1]}")

print("\n=== COLUMNAS ===")
print(df_poblacion.columns.tolist())

print("\n=== TIPOS DE DATOS ===")
print(df_poblacion.dtypes)

print("\n=== PRIMERAS 5 FILAS ===")
display(df_poblacion.head())

print("\n=== ÚLTIMAS 5 FILAS ===")
display(df_poblacion.tail())

=== DIMENSIONES DEL DATASET ===
Filas: 131502
Columnas: 8

=== COLUMNAS ===
['ANO', 'CODIGO_LOCALIDAD', 'NOMBRE_LOCALIDAD', 'SEXO', 'EDAD', 'CURSODEVIDA', 'GRUPOEDAD', 'POBLACION']

=== TIPOS DE DATOS ===
ANO                 int64
CODIGO_LOCALIDAD    int64
NOMBRE_LOCALIDAD      str
SEXO                  str
EDAD                int64
CURSODEVIDA           str
GRUPOEDAD             str
POBLACION           int64
dtype: object

=== PRIMERAS 5 FILAS ===


,ANO,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD,SEXO,EDAD,CURSODEVIDA,GRUPOEDAD,POBLACION
0,2005,0,Bogotá,Hombres,6,Infancia,00 a 11,67184
1,2005,0,Bogotá,Hombres,7,Infancia,00 a 11,68940
2,2005,0,Bogotá,Hombres,8,Infancia,00 a 11,70568
3,2005,0,Bogotá,Hombres,9,Infancia,00 a 11,71189
4,2005,0,Bogotá,Hombres,10,Infancia,00 a 11,70398



=== ÚLTIMAS 5 FILAS ===


,ANO,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD,SEXO,EDAD,CURSODEVIDA,GRUPOEDAD,POBLACION
131497,2035,20,Sumapaz,Mujeres,96,Vejez,60 o más,1
131498,2035,20,Sumapaz,Mujeres,97,Vejez,60 o más,0
131499,2035,20,Sumapaz,Mujeres,98,Vejez,60 o más,0
131500,2035,20,Sumapaz,Mujeres,99,Vejez,60 o más,0
131501,2035,20,Sumapaz,Mujeres,100,Vejez,60 o más,0


In [29]:
print("=== CÓDIGOS DE LOCALIDAD ===")
print(sorted(df_poblacion["CODIGO_LOCALIDAD"].dropna().unique()))

print("\n=== NOMBRES DE LOCALIDAD ===")
print(sorted(df_poblacion["NOMBRE_LOCALIDAD"].dropna().unique()))

print("\n=== CANTIDAD DE CÓDIGOS ÚNICOS ===")
print(df_poblacion["CODIGO_LOCALIDAD"].nunique())

print("\n=== CANTIDAD DE NOMBRES ÚNICOS ===")
print(df_poblacion["NOMBRE_LOCALIDAD"].nunique())

print("\n=== RELACIÓN CÓDIGO - LOCALIDAD ===")
display(
    df_poblacion[
        ["CODIGO_LOCALIDAD", "NOMBRE_LOCALIDAD"]
    ]
    .drop_duplicates()
    .sort_values("CODIGO_LOCALIDAD")
)

=== CÓDIGOS DE LOCALIDAD ===
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]

=== NOMBRES DE LOCALIDAD ===
['Antonio Nariño', 'Barrios Unidos', 'Bogotá', 'Bosa', 'Chapinero', 'Ciudad Bolívar', 'Engativá', 'Fontibón', 'Kennedy', 'La Candelaria', 'Los Mártires', 'Puente Aranda', 'Rafael Uribe Uribe', 'San Cristóbal', 'Santa Fe', 'Suba', 'Sumapaz', 'Teusaquillo', 'Tunjuelito', 'Usaquén', 'Usme']

=== CANTIDAD DE CÓDIGOS ÚNICOS ===
21

=== CANTIDAD DE NOMBRES ÚNICOS ===
21

=== RELACIÓN CÓDIGO - LOCALIDAD ===


,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD
0,0,Bogotá
6,1,Usaquén
12,2,Chapinero
18,3,Santa Fe
24,4,San Cristóbal
30,5,Usme
36,6,Tunjuelito
42,7,Bosa
48,8,Kennedy
54,9,Fontibón


In [31]:
print("=== AÑOS DISPONIBLES ===")

print(sorted(df_poblacion["ANO"].dropna().unique()))

print("\nAño mínimo:")
print(df_poblacion["ANO"].min())

print("\nAño máximo:")
print(df_poblacion["ANO"].max())

print("\nCantidad de años distintos:")
print(df_poblacion["ANO"].nunique())

=== AÑOS DISPONIBLES ===
[np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]

Año mínimo:
2005

Año máximo:
2035

Cantidad de años distintos:
31


In [32]:
print("=== VALORES NULOS POR COLUMNA ===")

nulos = df_poblacion.isna().sum()

porcentaje_nulos = (
    df_poblacion.isna().mean() * 100
).round(2)

validacion_nulos = (
    pd.DataFrame({
        "nulos": nulos,
        "porcentaje": porcentaje_nulos
    })
    .sort_values("porcentaje", ascending=False)
)

display(validacion_nulos)

=== VALORES NULOS POR COLUMNA ===


,nulos,porcentaje
ANO,0,0.0
CODIGO_LOCALIDAD,0,0.0
NOMBRE_LOCALIDAD,0,0.0
SEXO,0,0.0
EDAD,0,0.0
CURSODEVIDA,0,0.0
GRUPOEDAD,0,0.0
POBLACION,0,0.0


In [33]:
print("=== DUPLICADOS EXACTOS ===")

duplicados_exactos = df_poblacion.duplicated().sum()

print(f"Filas duplicadas exactas: {duplicados_exactos}")

porcentaje_duplicados = (
    duplicados_exactos / len(df_poblacion) * 100
)

print(
    f"Porcentaje de duplicados: "
    f"{porcentaje_duplicados:.4f}%"
)

=== DUPLICADOS EXACTOS ===
Filas duplicadas exactas: 0
Porcentaje de duplicados: 0.0000%


## Guardar versión raw con metadatos

In [10]:
def save_versioned_raw(df: pd.DataFrame, filename: str, suffix: str = 'v1') -> Path:
    output = RAW_DIR / f'{filename.stem}_{suffix}{filename.suffix}'
    df.to_csv(output, index=False)
    return output

# Ejemplo de uso:
# save_versioned_raw(df, Path('dataset.csv'), suffix='20260731')


## Guardar una versión procesada

In [ ]:
def save_processed_copy(df: pd.DataFrame, output_name: str) -> Path:
    destination = PROCESSED_DIR / output_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(destination, index=False)
    return destination

## Notas de ingesta — Demografía

- La fuente demográfica fue descargada y almacenada en
  `data/raw/DEMOGRAFIA/osb_demografia-poblacion-localidad.csv`.

- El archivo crudo debe conservarse sin modificaciones manuales para mantener
  la trazabilidad y reproducibilidad del análisis.

- El CSV utiliza `;` como separador y se carga mediante codificación
  `utf-8-sig`.

- Durante las pruebas iniciales se detectó un Byte Order Mark (BOM) asociado
  al encabezado `ANO`; la lectura con `utf-8-sig` permite interpretar
  correctamente el nombre de la columna sin modificar el archivo original.

- La carga completa produce 131.502 registros y 8 variables.

- Los metadatos de la fuente se almacenan por separado en `data/external`
  cuando corresponda.

- Este notebook se utiliza para probar y documentar el proceso de ingesta.
  Una vez estabilizada la lógica reutilizable, podrá trasladarse a
  `src/ingestion/`.

- Las transformaciones, correcciones o reglas de calidad no deben aplicarse
  sobre `data/raw`; deben realizarse en etapas posteriores del pipeline.